# ДЗ 5. Узагальнені табличні вирази (CTE), Materialized View

**Датасет:** Olist Brazilian E-Commerce, https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce

## Завдання 1. Setup і typed-схема Olist

In [1]:
%pip install -q pgserver psycopg2-binary sqlalchemy pandas kagglehub

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import pandas as pd
import pgserver
from sqlalchemy import create_engine, text

pg = pgserver.get_server('/tmp/hw5_pg', cleanup_mode='stop')
engine = create_engine(pg.get_uri().replace('postgresql://', 'postgresql+psycopg2://', 1), future=True)

with engine.connect() as conn:
    print(conn.execute(text('SELECT version()')).scalar_one())

def sql(q):
    with engine.begin() as conn:
        res = conn.exec_driver_sql(q)
        if res.returns_rows:
            return pd.DataFrame(res.fetchall(), columns=list(res.keys()))

def explain(q):
    print('\n'.join(sql(q).iloc[:, 0]))

PostgreSQL 16.2 on aarch64-apple-darwin23.5.0, compiled by Apple clang version 15.0.0 (clang-1500.0.40.1), 64-bit


In [3]:
import kagglehub

dataset_dir = kagglehub.dataset_download('olistbr/brazilian-ecommerce')
print(dataset_dir)

/Users/cumbainer/goit-rdb/goit-rdb-hw-03/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


/Users/cumbainer/.cache/kagglehub/datasets/olistbr/brazilian-ecommerce/versions/2


In [4]:
sql('DROP MATERIALIZED VIEW IF EXISTS mv_olist_customer_features CASCADE')
sql('DROP TABLE IF EXISTS hw5_referrals, olist_order_items, olist_orders, olist_customers, olist_products, olist_sellers CASCADE')

files = {
    'olist_customers_raw': 'olist_customers_dataset.csv',
    'olist_orders_raw': 'olist_orders_dataset.csv',
    'olist_order_items_raw': 'olist_order_items_dataset.csv',
    'olist_products_raw': 'olist_products_dataset.csv',
    'olist_sellers_raw': 'olist_sellers_dataset.csv',
}

for table_name, file_name in files.items():
    df = pd.read_csv(os.path.join(dataset_dir, file_name))
    df.to_sql(table_name, engine, if_exists='replace', index=False, chunksize=10_000)
    print(table_name, df.shape)

olist_customers_raw (99441, 5)


olist_orders_raw (99441, 8)


olist_order_items_raw (112650, 7)


olist_products_raw (32951, 9)
olist_sellers_raw (3095, 4)


In [5]:
sql("""
CREATE TABLE olist_customers (
    customer_id TEXT PRIMARY KEY,
    customer_unique_id TEXT NOT NULL,
    customer_zip_code_prefix INTEGER,
    customer_city TEXT,
    customer_state CHAR(2) NOT NULL
);

CREATE TABLE olist_orders (
    order_id TEXT PRIMARY KEY,
    customer_id TEXT NOT NULL REFERENCES olist_customers(customer_id),
    order_status TEXT NOT NULL,
    order_purchase_timestamp TIMESTAMP NOT NULL,
    order_approved_at TIMESTAMP,
    order_delivered_carrier_date TIMESTAMP,
    order_delivered_customer_date TIMESTAMP,
    order_estimated_delivery_date TIMESTAMP,
    CHECK (order_status IN (
        'created', 'approved', 'invoiced', 'processing',
        'shipped', 'delivered', 'unavailable', 'canceled'
    ))
);

CREATE TABLE olist_products (
    product_id TEXT PRIMARY KEY,
    product_category_name TEXT,
    product_name_length INTEGER CHECK (product_name_length IS NULL OR product_name_length >= 0),
    product_description_length INTEGER CHECK (product_description_length IS NULL OR product_description_length >= 0),
    product_photos_qty INTEGER CHECK (product_photos_qty IS NULL OR product_photos_qty >= 0),
    product_weight_g INTEGER CHECK (product_weight_g IS NULL OR product_weight_g >= 0),
    product_length_cm INTEGER CHECK (product_length_cm IS NULL OR product_length_cm >= 0),
    product_height_cm INTEGER CHECK (product_height_cm IS NULL OR product_height_cm >= 0),
    product_width_cm INTEGER CHECK (product_width_cm IS NULL OR product_width_cm >= 0)
);

CREATE TABLE olist_sellers (
    seller_id TEXT PRIMARY KEY,
    seller_zip_code_prefix INTEGER,
    seller_city TEXT,
    seller_state CHAR(2) NOT NULL
);

CREATE TABLE olist_order_items (
    order_id TEXT NOT NULL REFERENCES olist_orders(order_id),
    order_item_id INTEGER NOT NULL,
    product_id TEXT REFERENCES olist_products(product_id),
    seller_id TEXT REFERENCES olist_sellers(seller_id),
    shipping_limit_date TIMESTAMP,
    price NUMERIC(12, 2) CHECK (price IS NULL OR price >= 0),
    freight_value NUMERIC(12, 2) CHECK (freight_value IS NULL OR freight_value >= 0),
    PRIMARY KEY (order_id, order_item_id)
);
""")

In [6]:
sql("""
INSERT INTO olist_customers
SELECT customer_id, customer_unique_id, customer_zip_code_prefix::INTEGER, customer_city, customer_state::CHAR(2)
FROM olist_customers_raw;

INSERT INTO olist_orders
SELECT order_id, customer_id, order_status,
       order_purchase_timestamp::TIMESTAMP, order_approved_at::TIMESTAMP,
       order_delivered_carrier_date::TIMESTAMP, order_delivered_customer_date::TIMESTAMP,
       order_estimated_delivery_date::TIMESTAMP
FROM olist_orders_raw;

INSERT INTO olist_products
SELECT product_id, product_category_name,
       product_name_lenght::INTEGER, product_description_lenght::INTEGER,
       product_photos_qty::INTEGER, product_weight_g::INTEGER,
       product_length_cm::INTEGER, product_height_cm::INTEGER, product_width_cm::INTEGER
FROM olist_products_raw;

INSERT INTO olist_sellers
SELECT seller_id, seller_zip_code_prefix::INTEGER, seller_city, seller_state::CHAR(2)
FROM olist_sellers_raw;

INSERT INTO olist_order_items
SELECT order_id, order_item_id::INTEGER, product_id, seller_id,
       shipping_limit_date::TIMESTAMP, price::NUMERIC(12, 2), freight_value::NUMERIC(12, 2)
FROM olist_order_items_raw;

ANALYZE;
""")

In [7]:
sql("""
SELECT 'customers' AS table_name, COUNT(*) AS n_rows FROM olist_customers
UNION ALL SELECT 'orders', COUNT(*) FROM olist_orders
UNION ALL SELECT 'order_items', COUNT(*) FROM olist_order_items
UNION ALL SELECT 'products', COUNT(*) FROM olist_products
UNION ALL SELECT 'sellers', COUNT(*) FROM olist_sellers
ORDER BY table_name;
""")

,table_name,n_rows
0,customers,99441
1,order_items,112650
2,orders,99441
3,products,32951
4,sellers,3095


**Висновок:** кількість рядків у typed-таблицях збігається з raw-таблицями: customers 99 441, orders 99 441, order_items 112 650, products 32 951, sellers 3 095. Жоден рядок не відкинуто обмеженнями.

## Завдання 2. CTE-chain pipeline

In [8]:
sql("""
WITH
reference_date AS (
    -- Stage 0: stable reference date for historical dataset
    SELECT (MAX(order_purchase_timestamp)::DATE + 1) AS as_of_date
    FROM olist_orders
),
stage1_orders_total AS (
    -- Stage 1: orders + order_total + status
    SELECT
        o.customer_id,
        o.order_id,
        o.order_status,
        o.order_purchase_timestamp AS order_ts,
        o.order_purchase_timestamp::DATE AS order_date,
        COALESCE(SUM(COALESCE(oi.price, 0) + COALESCE(oi.freight_value, 0)), 0)::NUMERIC(12, 2) AS order_total
    FROM olist_orders AS o
    LEFT JOIN olist_order_items AS oi ON oi.order_id = o.order_id
    GROUP BY o.customer_id, o.order_id, o.order_status, o.order_purchase_timestamp
),
stage2_rfm AS (
    -- Stage 2: RFM features per customer
    SELECT
        s.customer_id,
        (rd.as_of_date - MAX(s.order_date))::INTEGER AS recency_days,
        COUNT(*) AS frequency,
        SUM(s.order_total)::NUMERIC(12, 2) AS monetary,
        MIN(s.order_date) AS first_order_date
    FROM stage1_orders_total AS s
    CROSS JOIN reference_date AS rd
    GROUP BY s.customer_id, rd.as_of_date
),
stage3_cohort AS (
    -- Stage 3: cohort quarter from first order date
    SELECT r.*, DATE_TRUNC('quarter', r.first_order_date)::DATE AS cohort_quarter
    FROM stage2_rfm AS r
),
stage4_rolling_aov AS (
    -- Stage 4: rolling 30-day average order value per customer
    SELECT
        s.customer_id,
        s.order_id,
        s.order_ts,
        s.order_total,
        AVG(s.order_total) OVER (
            PARTITION BY s.customer_id
            ORDER BY s.order_ts
            RANGE BETWEEN INTERVAL '30 days' PRECEDING AND CURRENT ROW
        )::NUMERIC(12, 2) AS avg_order_value_30d,
        ROW_NUMBER() OVER (
            PARTITION BY s.customer_id
            ORDER BY s.order_ts DESC, s.order_id DESC
        ) AS rn_recent
    FROM stage1_orders_total AS s
),
stage5_cancel_rate AS (
    -- Stage 5: cancellation rate per customer
    SELECT
        customer_id,
        AVG(CASE WHEN order_status = 'canceled' THEN 1.0 ELSE 0.0 END)::NUMERIC(6, 4) AS cancellation_rate
    FROM stage1_orders_total
    GROUP BY customer_id
),
stage6_category_revenue AS (
    -- Stage 6: revenue by customer and product category
    SELECT
        o.customer_id,
        COALESCE(p.product_category_name, '(unknown)') AS product_category_name,
        SUM(COALESCE(oi.price, 0) + COALESCE(oi.freight_value, 0))::NUMERIC(12, 2) AS category_revenue
    FROM olist_orders AS o
    JOIN olist_order_items AS oi ON oi.order_id = o.order_id
    LEFT JOIN olist_products AS p ON p.product_id = oi.product_id
    GROUP BY o.customer_id, COALESCE(p.product_category_name, '(unknown)')
),
stage7_preferred_category AS (
    -- Stage 7: preferred category per customer via top-1 ranking
    SELECT customer_id, product_category_name, category_revenue
    FROM (
        SELECT cr.*,
               ROW_NUMBER() OVER (
                   PARTITION BY cr.customer_id
                   ORDER BY cr.category_revenue DESC, cr.product_category_name
               ) AS rn
        FROM stage6_category_revenue AS cr
    ) AS ranked
    WHERE rn = 1
)
SELECT
    c.customer_id,
    c.customer_state,
    COALESCE(coh.recency_days, 9999) AS recency_days,
    COALESCE(coh.frequency, 0) AS frequency,
    COALESCE(coh.monetary, 0)::NUMERIC(12, 2) AS monetary,
    coh.first_order_date,
    coh.cohort_quarter,
    COALESCE(aov.avg_order_value_30d, 0)::NUMERIC(12, 2) AS avg_order_value_30d_recent,
    COALESCE(cancel.cancellation_rate, 0)::NUMERIC(6, 4) AS cancellation_rate,
    COALESCE(pc.product_category_name, '(none)') AS preferred_category,
    COALESCE(pc.category_revenue, 0)::NUMERIC(12, 2) AS preferred_category_revenue
FROM olist_customers AS c
LEFT JOIN stage3_cohort AS coh ON coh.customer_id = c.customer_id
LEFT JOIN stage4_rolling_aov AS aov ON aov.customer_id = c.customer_id AND aov.rn_recent = 1
LEFT JOIN stage5_cancel_rate AS cancel ON cancel.customer_id = c.customer_id
LEFT JOIN stage7_preferred_category AS pc ON pc.customer_id = c.customer_id
ORDER BY monetary DESC NULLS LAST, c.customer_id
LIMIT 20;
""")

,customer_id,customer_state,recency_days,frequency,monetary,first_order_date,cohort_quarter,avg_order_value_30d_recent,cancellation_rate,preferred_category,preferred_category_revenue
0,1617b1357756262bfa56ab541c47bc16,RJ,384,1,13664.08,2017-09-29,2017-07-01,13664.08,0.0000,telefonia_fixa,13664.08
1,ec5b2ba62e574342386871631fafd3fc,ES,95,1,7274.88,2018-07-15,2018-07-01,7274.88,0.0000,telefonia_fixa,7274.88
2,c6e2731c5b391845f6800c97401a43a9,MS,613,1,6929.31,2017-02-12,2017-01-01,6929.31,0.0000,utilidades_domesticas,6929.31
3,f48d464a0baaea338cb25f816991ab1f,ES,85,1,6922.21,2018-07-25,2018-07-01,6922.21,0.0000,pcs,6922.21
4,3fd6777bbce08a352fddd04e4a7cc8f6,SP,512,1,6726.66,2017-05-24,2017-04-01,6726.66,0.0000,artes,6726.66
5,05455dfa7cd02f13d132aa7a6a9729c6,MG,328,1,6081.54,2017-11-24,2017-10-01,6081.54,0.0000,agro_industria_e_comercio,6081.54
6,df55c14d1476a9a3467f131269c2477f,RJ,565,1,4950.34,2017-04-01,2017-04-01,4950.34,0.0000,eletroportateis,4950.34
7,e0a2412720e9ea4f26c1ac985f6a7358,GO,98,1,4809.44,2018-07-12,2018-07-01,4809.44,1.0000,cool_stuff,4809.44
8,24bbf5fd2f2e1b359ee7de94defc4a15,SP,548,1,4764.34,2017-04-18,2017-04-01,4764.34,0.0000,eletroportateis,4764.34
9,3d979689f636322c62418b6346b1c6d2,PB,118,1,4681.78,2018-06-22,2018-04-01,4681.78,0.0000,pcs,4681.78


**Пояснення:** pipeline складається з восьми named stages, кожен виконує одну трансформацію: reference date → order_total → RFM → cohort → rolling AOV → cancellation rate → category revenue → top-1 category. `reference_date` береться з датасету (`MAX(order_purchase_timestamp) + 1`), тому recency стабільна між запусками. Найбільший клієнт витратив 13 664 на `telefonia_fixa`; у топ-20 є клієнт із `cancellation_rate = 1.0`, тобто дороге, але скасоване замовлення. **Обмеження:** в Olist `customer_id` створюється для кожного замовлення, тому `frequency` майже завжди 1, а rolling AOV збігається з `order_total`. Для реальної моделі ті самі stages варто рахувати по `customer_unique_id`.

## Завдання 3. Recursive CTE для referral chain

In [9]:
sql("""
CREATE TABLE hw5_referrals (
    referrer_customer_id TEXT NOT NULL REFERENCES olist_customers(customer_id),
    referred_customer_id TEXT NOT NULL REFERENCES olist_customers(customer_id),
    referred_at TIMESTAMPTZ NOT NULL DEFAULT NOW(),
    PRIMARY KEY (referred_customer_id),
    CHECK (referrer_customer_id <> referred_customer_id)
);

WITH ordered_customers AS (
    SELECT customer_id, ROW_NUMBER() OVER (ORDER BY customer_id) AS rn
    FROM olist_customers
    ORDER BY customer_id
    LIMIT 10
)
INSERT INTO hw5_referrals (referrer_customer_id, referred_customer_id)
SELECT prev.customer_id, nxt.customer_id
FROM ordered_customers AS prev
JOIN ordered_customers AS nxt ON nxt.rn = prev.rn + 1
ON CONFLICT (referred_customer_id) DO NOTHING;
""")

In [10]:
sql("""
SELECT referrer_customer_id, referred_customer_id
FROM hw5_referrals
ORDER BY referrer_customer_id, referred_customer_id;
""")

,referrer_customer_id,referred_customer_id
0,00012a2ce6f8dcda20d059ce98491703,000161a058600d5901f007fab4c27140
1,000161a058600d5901f007fab4c27140,0001fd6190edaaf884bcaf3d49edf079
2,0001fd6190edaaf884bcaf3d49edf079,0002414f95344307404f0ace7a26f1d5
3,0002414f95344307404f0ace7a26f1d5,000379cdec625522490c315e70c7a9fb
4,000379cdec625522490c315e70c7a9fb,0004164d20a9e969af783496f3408652
5,0004164d20a9e969af783496f3408652,000419c5494106c306a97b5635748086
6,000419c5494106c306a97b5635748086,00046a560d407e99b969756e0b10f282
7,00046a560d407e99b969756e0b10f282,00050bf6e01e69d5c0fd612f1bcfb69c
8,00050bf6e01e69d5c0fd612f1bcfb69c,000598caf2ef4117407665ac33275130


In [11]:
sql("""
WITH RECURSIVE referral_depth AS (
    SELECT
        c.customer_id,
        1 AS depth,
        ARRAY[c.customer_id]::TEXT[] AS path_ids,
        c.customer_id::TEXT AS path_text
    FROM olist_customers AS c
    WHERE NOT EXISTS (
        SELECT 1 FROM hw5_referrals AS r WHERE r.referred_customer_id = c.customer_id
    )

    UNION ALL

    SELECT
        r.referred_customer_id,
        rd.depth + 1,
        rd.path_ids || r.referred_customer_id,
        rd.path_text || ' -> ' || r.referred_customer_id
    FROM hw5_referrals AS r
    JOIN referral_depth AS rd ON rd.customer_id = r.referrer_customer_id
    WHERE NOT r.referred_customer_id = ANY(rd.path_ids)
      AND rd.depth < 50
)
SELECT depth, COUNT(*) AS n_customers, LEFT(MAX(path_text), 120) AS sample_path
FROM referral_depth
GROUP BY depth
ORDER BY depth;
""")

,depth,n_customers,sample_path
0,1,99432,ffffe8b65bbe3087b653a978c870db99
1,2,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058...
2,3,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058...
3,4,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058...
4,5,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058...
5,6,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058...
6,7,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058...
7,8,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058...
8,9,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058...
9,10,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058...


**Пояснення:** синтетичний ланцюг із 10 клієнтів дає глибину від 1 до 10: 99 432 кореневих клієнтів (без referrer) на depth 1 і по одному клієнту на кожному наступному рівні. Anchor використовує `NOT EXISTS`, а recursive part переходить від referrer до referred. Термінацію гарантують два захисти: `NOT r.referred_customer_id = ANY(rd.path_ids)` не дає повторно відвідати вузол, а `rd.depth < 50` обмежує глибину навіть у разі помилки в даних.

## Завдання 4. Materialized View з CONCURRENT REFRESH

In [12]:
sql("""
CREATE MATERIALIZED VIEW mv_olist_customer_features AS
WITH
reference_date AS (
    -- Stage 0: stable reference date for historical dataset
    SELECT (MAX(order_purchase_timestamp)::DATE + 1) AS as_of_date
    FROM olist_orders
),
stage1_orders_total AS (
    -- Stage 1: orders + order_total + status
    SELECT
        o.customer_id,
        o.order_id,
        o.order_status,
        o.order_purchase_timestamp AS order_ts,
        o.order_purchase_timestamp::DATE AS order_date,
        COALESCE(SUM(COALESCE(oi.price, 0) + COALESCE(oi.freight_value, 0)), 0)::NUMERIC(12, 2) AS order_total
    FROM olist_orders AS o
    LEFT JOIN olist_order_items AS oi ON oi.order_id = o.order_id
    GROUP BY o.customer_id, o.order_id, o.order_status, o.order_purchase_timestamp
),
stage2_rfm AS (
    -- Stage 2: RFM features per customer
    SELECT
        s.customer_id,
        (rd.as_of_date - MAX(s.order_date))::INTEGER AS recency_days,
        COUNT(*) AS frequency,
        SUM(s.order_total)::NUMERIC(12, 2) AS monetary,
        MIN(s.order_date) AS first_order_date
    FROM stage1_orders_total AS s
    CROSS JOIN reference_date AS rd
    GROUP BY s.customer_id, rd.as_of_date
),
stage3_cohort AS (
    -- Stage 3: cohort quarter from first order date
    SELECT r.*, DATE_TRUNC('quarter', r.first_order_date)::DATE AS cohort_quarter
    FROM stage2_rfm AS r
),
stage4_rolling_aov AS (
    -- Stage 4: rolling 30-day average order value per customer
    SELECT
        s.customer_id,
        s.order_id,
        s.order_ts,
        s.order_total,
        AVG(s.order_total) OVER (
            PARTITION BY s.customer_id
            ORDER BY s.order_ts
            RANGE BETWEEN INTERVAL '30 days' PRECEDING AND CURRENT ROW
        )::NUMERIC(12, 2) AS avg_order_value_30d,
        ROW_NUMBER() OVER (
            PARTITION BY s.customer_id
            ORDER BY s.order_ts DESC, s.order_id DESC
        ) AS rn_recent
    FROM stage1_orders_total AS s
),
stage5_cancel_rate AS (
    -- Stage 5: cancellation rate per customer
    SELECT
        customer_id,
        AVG(CASE WHEN order_status = 'canceled' THEN 1.0 ELSE 0.0 END)::NUMERIC(6, 4) AS cancellation_rate
    FROM stage1_orders_total
    GROUP BY customer_id
),
stage6_category_revenue AS (
    -- Stage 6: revenue by customer and product category
    SELECT
        o.customer_id,
        COALESCE(p.product_category_name, '(unknown)') AS product_category_name,
        SUM(COALESCE(oi.price, 0) + COALESCE(oi.freight_value, 0))::NUMERIC(12, 2) AS category_revenue
    FROM olist_orders AS o
    JOIN olist_order_items AS oi ON oi.order_id = o.order_id
    LEFT JOIN olist_products AS p ON p.product_id = oi.product_id
    GROUP BY o.customer_id, COALESCE(p.product_category_name, '(unknown)')
),
stage7_preferred_category AS (
    -- Stage 7: preferred category per customer via top-1 ranking
    SELECT customer_id, product_category_name, category_revenue
    FROM (
        SELECT cr.*,
               ROW_NUMBER() OVER (
                   PARTITION BY cr.customer_id
                   ORDER BY cr.category_revenue DESC, cr.product_category_name
               ) AS rn
        FROM stage6_category_revenue AS cr
    ) AS ranked
    WHERE rn = 1
)
SELECT
    c.customer_id,
    c.customer_state,
    COALESCE(coh.recency_days, 9999) AS recency_days,
    COALESCE(coh.frequency, 0) AS frequency,
    COALESCE(coh.monetary, 0)::NUMERIC(12, 2) AS monetary,
    coh.first_order_date,
    coh.cohort_quarter,
    COALESCE(aov.avg_order_value_30d, 0)::NUMERIC(12, 2) AS avg_order_value_30d_recent,
    COALESCE(cancel.cancellation_rate, 0)::NUMERIC(6, 4) AS cancellation_rate,
    COALESCE(pc.product_category_name, '(none)') AS preferred_category,
    COALESCE(pc.category_revenue, 0)::NUMERIC(12, 2) AS preferred_category_revenue
FROM olist_customers AS c
LEFT JOIN stage3_cohort AS coh ON coh.customer_id = c.customer_id
LEFT JOIN stage4_rolling_aov AS aov ON aov.customer_id = c.customer_id AND aov.rn_recent = 1
LEFT JOIN stage5_cancel_rate AS cancel ON cancel.customer_id = c.customer_id
LEFT JOIN stage7_preferred_category AS pc ON pc.customer_id = c.customer_id;

CREATE UNIQUE INDEX idx_mv_olist_customer_features_customer_id
ON mv_olist_customer_features (customer_id);
""")

In [13]:
sql("""
SELECT COUNT(*) AS n_rows FROM mv_olist_customer_features;
""")

,n_rows
0,99441


In [14]:
sql("""
SELECT *
FROM mv_olist_customer_features
ORDER BY monetary DESC NULLS LAST, customer_id
LIMIT 5;
""")

,customer_id,customer_state,recency_days,frequency,monetary,first_order_date,cohort_quarter,avg_order_value_30d_recent,cancellation_rate,preferred_category,preferred_category_revenue
0,1617b1357756262bfa56ab541c47bc16,RJ,384,1,13664.08,2017-09-29,2017-07-01,13664.08,0.0000,telefonia_fixa,13664.08
1,ec5b2ba62e574342386871631fafd3fc,ES,95,1,7274.88,2018-07-15,2018-07-01,7274.88,0.0000,telefonia_fixa,7274.88
2,c6e2731c5b391845f6800c97401a43a9,MS,613,1,6929.31,2017-02-12,2017-01-01,6929.31,0.0000,utilidades_domesticas,6929.31
3,f48d464a0baaea338cb25f816991ab1f,ES,85,1,6922.21,2018-07-25,2018-07-01,6922.21,0.0000,pcs,6922.21
4,3fd6777bbce08a352fddd04e4a7cc8f6,SP,512,1,6726.66,2017-05-24,2017-04-01,6726.66,0.0000,artes,6726.66


In [15]:
sql("""
REFRESH MATERIALIZED VIEW CONCURRENTLY mv_olist_customer_features;
""")

In [16]:
sql("""
SELECT COUNT(*) AS n_rows_after_refresh FROM mv_olist_customer_features;
""")

,n_rows_after_refresh
0,99441


**Пояснення:** MV містить повний pipeline (усі 11 features) для 99 441 клієнта. `UNIQUE INDEX` на `customer_id` дозволив виконати `REFRESH MATERIALIZED VIEW CONCURRENTLY` без помилки; після refresh кількість рядків не змінилась. CONCURRENTLY перебудовує дані у фоні і не блокує читання MV на час оновлення.

## Завдання 5. EXPLAIN ANALYZE

### 5.1. Повне переобчислення CTE-pipeline

In [17]:
explain("""
EXPLAIN (ANALYZE, BUFFERS)
WITH
reference_date AS (
    -- Stage 0: stable reference date for historical dataset
    SELECT (MAX(order_purchase_timestamp)::DATE + 1) AS as_of_date
    FROM olist_orders
),
stage1_orders_total AS (
    -- Stage 1: orders + order_total + status
    SELECT
        o.customer_id,
        o.order_id,
        o.order_status,
        o.order_purchase_timestamp AS order_ts,
        o.order_purchase_timestamp::DATE AS order_date,
        COALESCE(SUM(COALESCE(oi.price, 0) + COALESCE(oi.freight_value, 0)), 0)::NUMERIC(12, 2) AS order_total
    FROM olist_orders AS o
    LEFT JOIN olist_order_items AS oi ON oi.order_id = o.order_id
    GROUP BY o.customer_id, o.order_id, o.order_status, o.order_purchase_timestamp
),
stage2_rfm AS (
    -- Stage 2: RFM features per customer
    SELECT
        s.customer_id,
        (rd.as_of_date - MAX(s.order_date))::INTEGER AS recency_days,
        COUNT(*) AS frequency,
        SUM(s.order_total)::NUMERIC(12, 2) AS monetary,
        MIN(s.order_date) AS first_order_date
    FROM stage1_orders_total AS s
    CROSS JOIN reference_date AS rd
    GROUP BY s.customer_id, rd.as_of_date
),
stage3_cohort AS (
    -- Stage 3: cohort quarter from first order date
    SELECT r.*, DATE_TRUNC('quarter', r.first_order_date)::DATE AS cohort_quarter
    FROM stage2_rfm AS r
),
stage4_rolling_aov AS (
    -- Stage 4: rolling 30-day average order value per customer
    SELECT
        s.customer_id,
        s.order_id,
        s.order_ts,
        s.order_total,
        AVG(s.order_total) OVER (
            PARTITION BY s.customer_id
            ORDER BY s.order_ts
            RANGE BETWEEN INTERVAL '30 days' PRECEDING AND CURRENT ROW
        )::NUMERIC(12, 2) AS avg_order_value_30d,
        ROW_NUMBER() OVER (
            PARTITION BY s.customer_id
            ORDER BY s.order_ts DESC, s.order_id DESC
        ) AS rn_recent
    FROM stage1_orders_total AS s
),
stage5_cancel_rate AS (
    -- Stage 5: cancellation rate per customer
    SELECT
        customer_id,
        AVG(CASE WHEN order_status = 'canceled' THEN 1.0 ELSE 0.0 END)::NUMERIC(6, 4) AS cancellation_rate
    FROM stage1_orders_total
    GROUP BY customer_id
),
stage6_category_revenue AS (
    -- Stage 6: revenue by customer and product category
    SELECT
        o.customer_id,
        COALESCE(p.product_category_name, '(unknown)') AS product_category_name,
        SUM(COALESCE(oi.price, 0) + COALESCE(oi.freight_value, 0))::NUMERIC(12, 2) AS category_revenue
    FROM olist_orders AS o
    JOIN olist_order_items AS oi ON oi.order_id = o.order_id
    LEFT JOIN olist_products AS p ON p.product_id = oi.product_id
    GROUP BY o.customer_id, COALESCE(p.product_category_name, '(unknown)')
),
stage7_preferred_category AS (
    -- Stage 7: preferred category per customer via top-1 ranking
    SELECT customer_id, product_category_name, category_revenue
    FROM (
        SELECT cr.*,
               ROW_NUMBER() OVER (
                   PARTITION BY cr.customer_id
                   ORDER BY cr.category_revenue DESC, cr.product_category_name
               ) AS rn
        FROM stage6_category_revenue AS cr
    ) AS ranked
    WHERE rn = 1
)
SELECT
    c.customer_id,
    c.customer_state,
    COALESCE(coh.recency_days, 9999) AS recency_days,
    COALESCE(coh.frequency, 0) AS frequency,
    COALESCE(coh.monetary, 0)::NUMERIC(12, 2) AS monetary,
    coh.first_order_date,
    coh.cohort_quarter,
    COALESCE(aov.avg_order_value_30d, 0)::NUMERIC(12, 2) AS avg_order_value_30d_recent,
    COALESCE(cancel.cancellation_rate, 0)::NUMERIC(6, 4) AS cancellation_rate,
    COALESCE(pc.product_category_name, '(none)') AS preferred_category,
    COALESCE(pc.category_revenue, 0)::NUMERIC(12, 2) AS preferred_category_revenue
FROM olist_customers AS c
LEFT JOIN stage3_cohort AS coh ON coh.customer_id = c.customer_id
LEFT JOIN stage4_rolling_aov AS aov ON aov.customer_id = c.customer_id AND aov.rn_recent = 1
LEFT JOIN stage5_cancel_rate AS cancel ON cancel.customer_id = c.customer_id
LEFT JOIN stage7_preferred_category AS pc ON pc.customer_id = c.customer_id
ORDER BY monetary DESC NULLS LAST, c.customer_id
LIMIT 10;
""")

Limit  (cost=115110.60..115110.63 rows=10 width=150) (actual time=538.865..539.068 rows=10 loops=1)
  Buffers: shared hit=111711, temp read=8438 written=8587
  CTE stage1_orders_total
    ->  GroupAggregate  (cost=0.83..27069.92 rows=99441 width=103) (actual time=0.014..66.579 rows=99441 loops=1)
          Group Key: o_1.order_id
          Buffers: shared hit=103453
          ->  Merge Left Join  (cost=0.83..24484.83 rows=112650 width=95) (actual time=0.007..39.843 rows=113425 loops=1)
                Merge Cond: (o_1.order_id = oi_1.order_id)
                Buffers: shared hit=103453
                ->  Index Scan using olist_orders_pkey on olist_orders o_1  (cost=0.42..12563.81 rows=99441 width=83) (actual time=0.003..19.317 rows=99441 loops=1)
                      Buffers: shared hit=100347
                ->  Index Scan using olist_order_items_pkey on olist_order_items oi_1  (cost=0.42..10264.29 rows=112650 width=45) (actual time=0.002..5.595 rows=112650 loops=1)
                

### 5.2. RFM-частина pipeline (stages 0–2)

In [18]:
explain("""
EXPLAIN (ANALYZE, BUFFERS)
WITH
reference_date AS (
    SELECT (MAX(order_purchase_timestamp)::DATE + 1) AS as_of_date
    FROM olist_orders
),
stage1_orders_total AS (
    SELECT o.customer_id, o.order_id, o.order_status,
           o.order_purchase_timestamp AS order_ts,
           o.order_purchase_timestamp::DATE AS order_date,
           COALESCE(SUM(COALESCE(oi.price, 0) + COALESCE(oi.freight_value, 0)), 0)::NUMERIC(12, 2) AS order_total
    FROM olist_orders AS o
    LEFT JOIN olist_order_items AS oi ON oi.order_id = o.order_id
    GROUP BY o.customer_id, o.order_id, o.order_status, o.order_purchase_timestamp
),
stage2_rfm AS (
    SELECT s.customer_id,
           (rd.as_of_date - MAX(s.order_date))::INTEGER AS recency_days,
           COUNT(*) AS frequency,
           SUM(s.order_total)::NUMERIC(12, 2) AS monetary,
           MIN(s.order_date) AS first_order_date
    FROM stage1_orders_total AS s
    CROSS JOIN reference_date AS rd
    GROUP BY s.customer_id, rd.as_of_date
)
SELECT *
FROM stage2_rfm
ORDER BY monetary DESC NULLS LAST
LIMIT 10;
""")

Limit  (cost=33611.21..33611.23 rows=10 width=65) (actual time=126.415..126.417 rows=10 loops=1)
  Buffers: shared hit=105261, temp read=961 written=1696
  ->  Sort  (cost=33611.21..33611.71 rows=200 width=65) (actual time=126.414..126.416 rows=10 loops=1)
        Sort Key: stage2_rfm.monetary DESC NULLS LAST
        Sort Method: top-N heapsort  Memory: 27kB
        Buffers: shared hit=105261, temp read=961 written=1696
        ->  Subquery Scan on stage2_rfm  (cost=33601.39..33606.89 rows=200 width=65) (actual time=93.380..120.305 rows=99441 loops=1)
              Buffers: shared hit=105261, temp read=961 written=1696
              ->  HashAggregate  (cost=33601.39..33604.89 rows=200 width=69) (actual time=93.379..116.958 rows=99441 loops=1)
                    Group Key: o.customer_id, (((max(olist_orders.order_purchase_timestamp))::date + 1))
                    Batches: 21  Memory Usage: 8265kB  Disk Usage: 7400kB
                    Buffers: shared hit=105261, temp read=961 writte

### 5.3. Lookup із MV за customer_id

In [19]:
explain("""
EXPLAIN (ANALYZE, BUFFERS)
WITH target_customer AS (
    SELECT customer_id FROM olist_customers ORDER BY customer_id LIMIT 1
)
SELECT mv.*
FROM mv_olist_customer_features AS mv
JOIN target_customer AS t ON t.customer_id = mv.customer_id;
""")

Nested Loop  (cost=0.83..8.96 rows=1 width=158) (actual time=0.032..0.033 rows=1 loops=1)
  Buffers: shared hit=5 read=3
  ->  Limit  (cost=0.42..0.53 rows=1 width=33) (actual time=0.020..0.020 rows=1 loops=1)
        Buffers: shared hit=4
        ->  Index Only Scan using olist_customers_pkey on olist_customers  (cost=0.42..11127.89 rows=99441 width=33) (actual time=0.019..0.019 rows=1 loops=1)
              Heap Fetches: 1
              Buffers: shared hit=4
  ->  Index Scan using idx_mv_olist_customer_features_customer_id on mv_olist_customer_features mv  (cost=0.42..8.44 rows=1 width=158) (actual time=0.012..0.012 rows=1 loops=1)
        Index Cond: (customer_id = olist_customers.customer_id)
        Buffers: shared hit=1 read=3
Planning Time: 0.078 ms
Execution Time: 0.040 ms


### 5.4. Top-N із MV з індексом на monetary

In [20]:
sql("""
CREATE INDEX IF NOT EXISTS idx_mv_olist_customer_features_monetary ON mv_olist_customer_features (monetary DESC NULLS LAST);
""")

In [21]:
explain("""
EXPLAIN (ANALYZE, BUFFERS)
SELECT *
FROM mv_olist_customer_features
ORDER BY monetary DESC NULLS LAST
LIMIT 10;
""")

Limit  (cost=0.42..1.32 rows=10 width=158) (actual time=0.007..0.014 rows=10 loops=1)
  Buffers: shared hit=10 read=3
  ->  Index Scan using idx_mv_olist_customer_features_monetary on mv_olist_customer_features  (cost=0.42..8996.03 rows=99441 width=158) (actual time=0.007..0.013 rows=10 loops=1)
        Buffers: shared hit=10 read=3
Planning:
  Buffers: shared hit=15 read=1
Planning Time: 0.104 ms
Execution Time: 0.019 ms


**Порівняння планів:** повне переобчислення pipeline займає **≈545 ms**, лише RFM-частина — **≈124 ms**, lookup із MV за `customer_id` — **0,04 ms** (`Index Scan using idx_mv_olist_customer_features_customer_id`), top-10 за `monetary` — **0,02 ms** (`Index Scan using idx_mv_olist_customer_features_monetary`, без Sort). У плані pipeline видно `Sort Method: external merge` на диску (до 9,4 MB), `HashAggregate` з `Batches: 21` і spill на диск, два `WindowAgg` для rolling AOV і ще один `WindowAgg` для top-1 категорії. Індекс створено як `monetary DESC NULLS LAST`, щоб він збігався з `ORDER BY`; індекс просто `DESC` планувальник не використав би.

## Reflection

За EXPLAIN ANALYZE найдорожчими є дві гілки з window functions. Rolling AOV (stage 4) потребує повного `Sort` 99 441 замовлень, який іде у `external merge` на диск (9,4 MB), а потім двох `WindowAgg`: ця гілка готова лише на ~177 ms. Preferred category (stages 6–7) робить три JOIN-и, `GroupAggregate`, `Incremental Sort` і `WindowAgg` з `row_number`, ще ~110 ms. Додатково `HashAggregate` для RFM і cancellation rate розливається на диск (`Batches: 21`). Загалом повний pipeline виконується ≈545 ms, а читання з MV — 0,02–0,04 ms через `Index Scan`, тобто приблизно в 10 000 разів швидше.

Кешувати ці features через MV варто: вони потрібні для навчання і scoring, а дані замовлень змінюються не щохвилини. Я обрав би щодобовий `REFRESH CONCURRENTLY` після нічного завантаження, а on-demand refresh — перед навчанням моделі. Звичайний VIEW правильніший, коли потрібна свіжість до секунди або запит дешевий, наприклад статус конкретного замовлення для служби підтримки, або коли features ще змінюються під час експериментів.

Recursive CTE завершується, коли recursive part перестає повертати нові рядки. Якщо в графі є цикл A → B → A, без захисту рекурсія була б нескінченною. Тут її зупиняє перевірка `NOT next_id = ANY(path_ids)`, яка не дозволяє повернутися до A, і depth guard `depth < 50`; у PostgreSQL 14+ те саме дає `CYCLE customer_id SET is_cycle USING path`.

Для цього pipeline головний trade-off — freshness проти recompute-cost: повне переобчислення коштує пів секунди CPU і диск на кожен запит, а MV дає миттєвий lookup ціною застарілості до одного refresh-циклу. Storage-cost MV (≈100 тис. рядків) несуттєвий. Для навчання моделі також важливо фіксувати момент refresh, щоб знати, станом на яку дату пораховано features, і не змішати версії даних між train і scoring.